# Basic MCMC
Part of the Bayesian neural networks via MCMC: a Python-based tutorial

This section of the tutorial covers the development of a basic MCMC algorithm.
This example takes inspiration from [this blog post](https://towardsdatascience.com/bayesian-inference-and-markov-chain-monte-carlo-sampling-in-python-bada1beabca7) with some simplifications and notation updated to align with the accompanying tutorial.

### Imports

In [ ]:
import numpy as np
from numpy import random
from tqdm import tqdm
from scipy import stats
# visulisation function
from functions.visualisations import histogram_trace

## Define the functions required for MCMC sampling

We will start with the simplest example, sampling the posterior of a single parameter.
In this example, we are trying to obtain information about the probability of success given some data of `k` successes in `n` trials using an uninformative prior.

This type of problem is represented by a binomial distribution and we will be solving for the posterior distribution of parameter `p` (probability of success) in this distribution.

First, we need to define the likelihood function given our data (`k`,`n`).

In [4]:
# First define our likelihood function which will be dependent on provided `data`
def generate_likelihood(k, n):
    '''
    Given the data of k successes in n trials, return a likelihood function which 
    evaluates the probability that a single success (p) is query_prob for any given 
    query_prob (between 0 and 1).
    This is defined by a binomial distribution. 
    See: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html
    L(p|k, n) = C(n, k) * p^k * (1-p)^(n - k)
    '''
    def likelihood(query_prob):
        print(query_prob)
        print("k: ", k)
        print("n: ", n)
        print("binom.pmf: ", stats.binom.pmf(k, n, query_prob))
        return stats.binom.pmf(k, n, query_prob)
    return likelihood

In [5]:
generate_likelihood(5, 10)(0.5)

0.5
k:  5
n:  10
binom.pmf:  0.24609375


np.float64(0.24609375)

## Sample using MCMC

- Create the MCMC loop and sample the posterior distribution
- We will use an un-informative (changed to un-informative, as the prior is uniform), uniform prior - $Pr(p) = 1$ for $p \in [0,1]$
###### every value p can take between 0 and 1 is equally likely, probability that p is between 0.2 and 0.3 is as same as the probability between 0.7 and 0.8 being 0.1%
- With this symmetric prior(no directional bias)  and the proposal `p` between 0 and 1, the contribution of the proposal distribution (which suggests a new candidate value xi for the next sample, given the current sample x') to the MH acceptance ratio is always:

$$\frac{q(x_i \mid x')}{q(x' \mid x_i)} = 1$$

- We therefore only need to compare the likelihood of the proposed and current sample 
$$ \frac{P(x')}{P(x_i)} = ?$$

In [6]:
## MCMC Settings and Setup
n_samples = 10000 # number of samples to draw from the posterior
burn_in = 2500 # number of samples to discard before recording draws from the posterior
# MCMC chains often start with poor or biased samples as the algorithm needs time to explore and
# converge to the true posterior. 
# Here, the first 2500 samples are thrown away and only the remaining 7500 samples are kept for analysis.

# specify our `data` for this example ensuring k <= n
binom_k = 50 # number of successes
binom_n = 100 # in n trials

x = random.uniform(0, 1) # initialise a value of x0
count = 0 # count the number of accepted samples

In [ ]:
# first, given the `data` provided we need to create our likelihood function
likelihood_function = generate_likelihood(binom_k, binom_n)

# create an array of NaNs to fill with our samples
p_posterior = np.full(n_samples, np.nan) 

print('Generating {} MCMC samples from the posterior:'.format(n_samples))
# now we can start the MCMC sampling loop
for ii in tqdm(np.arange(n_samples)):
    # Sample a value uniformly from 0 to 1 as a proposal
    x_new = random.uniform(0, 1)

    # Calculate the Metrpolis-Hastings acceptance probability based on the prior 
    # (can be ignored in this case) and likelihood
    prior_ratio = 1 # for this simple example as discussed above
    likelihood_ratio = likelihood_function(x_new) / likelihood_function(x)
    alpha = np.min([1, likelihood_ratio * prior_ratio])

    # Here we use a random draw from a uniform distribution between 0 and 1 as a 
    # method of accepting the new proposal with a probability of alpha
    # (i.e., accept if u < alpha)
    u = random.uniform(0, 1)
    if u < alpha:
        x = x_new # then update the current sample to the proposal for the next iteration
        count += 1 # add to the count of accepted samples

    # Store the current sample
    p_posterior[ii] = x
print('Acceptance rate: {:.2f}%'.format(count / n_samples * 100))

# Print all posterior samples
print("All posterior samples:")
print(p_posterior)

print('Done')
# p_posterior contains the MCMC samples from the posterior distribution of p given the values of k and n.


Generating 10000 MCMC samples from the posterior:


  1%|▏         | 132/10000 [00:00<00:08, 1233.34it/s]

0.5971739356528267
k:  50
n:  100
binom.pmf:  0.011608451251409989
0.0736086144943634
k:  50
n:  100
binom.pmf:  4.8979175258482595e-30
0.09839652194736148
k:  50
n:  100
binom.pmf:  2.5329126162963337e-24
0.5971739356528267
k:  50
n:  100
binom.pmf:  0.011608451251409989
0.7424755558039594
k:  50
n:  100
binom.pmf:  1.1991812874026838e-07
0.5971739356528267
k:  50
n:  100
binom.pmf:  0.011608451251409989
0.42562973209858657
k:  50
n:  100
binom.pmf:  0.026004549255400383
0.5971739356528267
k:  50
n:  100
binom.pmf:  0.011608451251409989
0.8574343131262294
k:  50
n:  100
binom.pmf:  2.3155140215197914e-17
0.42562973209858657
k:  50
n:  100
binom.pmf:  0.026004549255400383
0.3565094943665448
k:  50
n:  100
binom.pmf:  0.0010827198439039834
0.42562973209858657
k:  50
n:  100
binom.pmf:  0.026004549255400383
0.6553086885709796
k:  50
n:  100
binom.pmf:  0.0004985021522524384
0.42562973209858657
k:  50
n:  100
binom.pmf:  0.026004549255400383
0.10237574611995015
k:  50
n:  100
binom.pmf:  

  5%|▍         | 476/10000 [00:00<00:06, 1448.24it/s]

binom.pmf:  0.07873469368501176
0.08021831969522075
k:  50
n:  100
binom.pmf:  2.5221430244353506e-28
0.5073463653812201
k:  50
n:  100
binom.pmf:  0.07873469368501176
0.7480882269699773
k:  50
n:  100
binom.pmf:  5.806549663342178e-08
0.5073463653812201
k:  50
n:  100
binom.pmf:  0.07873469368501176
0.4118707494876922
k:  50
n:  100
binom.pmf:  0.016426324074993233
0.5073463653812201
k:  50
n:  100
binom.pmf:  0.07873469368501176
0.9961139853955917
k:  50
n:  100
binom.pmf:  2.480431352363086e-92
0.4118707494876922
k:  50
n:  100
binom.pmf:  0.016426324074993233
0.7874096600148642
k:  50
n:  100
binom.pmf:  1.5526270789234203e-10
0.4118707494876922
k:  50
n:  100
binom.pmf:  0.016426324074993233
0.7171291329665592
k:  50
n:  100
binom.pmf:  2.3074552243881432e-06
0.4118707494876922
k:  50
n:  100
binom.pmf:  0.016426324074993233
0.4115617475270621
k:  50
n:  100
binom.pmf:  0.016242351056266424
0.4118707494876922
k:  50
n:  100
binom.pmf:  0.016426324074993233
0.7504843835157314
k:  5

...

0.46644155423286227
k:  50
n:  100
binom.pmf:  0.06350611414342003
0.5710936031186732
k:  50
n:  100
binom.pmf:  0.028664532745973584
0.48385520225258927
k:  50
n:  100
binom.pmf:  0.07554441647315692
0.46644155423286227
k:  50
n:  100
binom.pmf:  0.06350611414342003
0.5269102665532148
k:  50
n:  100
binom.pmf:  0.0688435277281149
0.48385520225258927
k:  50
n:  100
binom.pmf:  0.07554441647315692
0.26015015967278
k:  50
n:  100
binom.pmf:  1.6681610244287708e-07
0.5269102665532148
k:  50
n:  100
binom.pmf:  0.0688435277281149
0.2317890920258019
k:  50
n:  100
binom.pmf:  3.4082158936796515e-09
0.5269102665532148
k:  50
n:  100
binom.pmf:  0.0688435277281149
0.38726088461586017
k:  50
n:  100
binom.pmf:  0.0058588965634929005
0.5269102665532148
k:  50
n:  100
binom.pmf:  0.0688435277281149
0.33440894626705686
k:  50
n:  100
binom.pmf:  0.0002388482563168157
0.5269102665532148
k:  50
n:  100
binom.pmf:  0.0688435277281149
0.5403868566954343
k:  50
n:  100
binom.pmf:  0.05737395141112088


 99%|█████████▉| 9941/10000 [00:08<00:00, 824.36it/s]

0.49598866705495237
k:  50
n:  100
binom.pmf:  0.07933351009078185
0.6112256743637167
k:  50
n:  100
binom.pmf:  0.006292339754137758
0.4907610209606994
k:  50
n:  100
binom.pmf:  0.07824181420569817
0.6169016770351371
k:  50
n:  100
binom.pmf:  0.004788108631300712
0.4907610209606994
k:  50
n:  100
binom.pmf:  0.07824181420569817
0.6145196815857977
k:  50
n:  100
binom.pmf:  0.005379652271243435
0.4907610209606994
k:  50
n:  100
binom.pmf:  0.07824181420569817
0.8637340628068534
k:  50
n:  100
binom.pmf:  3.4851513123758366e-18
0.4907610209606994
k:  50
n:  100
binom.pmf:  0.07824181420569817
0.5792566766125827
k:  50
n:  100
binom.pmf:  0.022298105943982995
0.4907610209606994
k:  50
n:  100
binom.pmf:  0.07824181420569817
0.1812996434106825
k:  50
n:  100
binom.pmf:  3.8014596265270983e-13
0.4907610209606994
k:  50
n:  100
binom.pmf:  0.07824181420569817
0.41838852602135135
k:  50
n:  100
binom.pmf:  0.020629584665782824
0.4907610209606994
k:  50
n:  100
binom.pmf:  0.078241814205698

100%|██████████| 10000/10000 [00:09<00:00, 1106.00it/s]

Acceptance rate: 15.82%
All posterior samples:
[0.59717394 0.59717394 0.59717394 ... 0.52511132 0.52511132 0.52511132]
Done


## Analyse the results
Plot the posterior distribution and trace

In [ ]:
# print(results, 'results')
per_accept = (count/n_samples)*100
print('{:.3f} % accepted'.format(per_accept))
posterior_mean = np.mean(p_posterior[burn_in:])
print('{:.3f} mean value of posterior'.format(posterior_mean))

15.820 % accepted
0.503 mean value of posterior


In [ ]:
histogram_trace(
    p_posterior, 
    true_posterior = np.vstack([np.linspace(0,1,1000),likelihood_function(np.linspace(0,1,1000))]).T, 
    burn_in = burn_in,
    title='Posterior - p',
    param_name = 'p',
    # fname='figures/02-Basic-MCMC'  
)
# np.linspace(0,1,1000) creates an array of 1000 evenly spaced values between 0 and 1, 
# which are used as input to the likelihood function.
# The sampler proposes many values of p in [0,1] uniformly at random, 
# but only accepts those that are more li(or with some probability if they are less likely).